In [7]:
# ============================================================
# Cellule 1 — Setup GPU + données + tokenizer + char vocab (P3)
# ============================================================

# 0) Imports avec install si manquant (top-level pour que spm/CHRF soient globaux)
import subprocess, sys
try:
    import sentencepiece as spm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "sentencepiece"],
                   check=True, timeout=60)
    import sentencepiece as spm

try:
    from sacrebleu.metrics import CHRF
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "sacrebleu"],
                   check=True, timeout=60)
    from sacrebleu.metrics import CHRF

# 1) Imports + verif GPU
import os, re, json, hashlib, math, random, time, csv
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", device,
      f"({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)"
      if device.type == "cuda" else "")
assert device.type == "cuda", "GPU non actif -> Settings -> Accelerator -> GPU T4 x2"

# 2) Chemins (structure doublée confirmée par os.walk)
DATA_ROOT = Path("/kaggle/input/datasets/cristoahouandjinou/p3-common-base")
CSV_DIR   = DATA_ROOT / "content_clean" / "content_clean"
SPM_PATH  = str(DATA_ROOT / "models" / "models" / "spm_fr.model")
CHAR_JSON = DATA_ROOT / "char2id.json"   # n'existe pas -> reconstruit depuis train
assert CSV_DIR.exists(), f"CSV_DIR introuvable : {CSV_DIR}"
assert Path(SPM_PATH).exists(), f"SPM_PATH introuvable : {SPM_PATH}"

# 3) Constantes P3
FON_COL, FR_COL = "fon", "fr"
MAX_SRC_WORDS, MAX_TGT_WORDS = 50, 60
MAX_SRC_W, MAX_TGT_T, MAX_WORD_LEN = 70, 80, 20
PAD, UNK = 0, 1

# 4) Empreintes attendues (integrite du upload)
EXPECTED_SHA = {"train": "dac241eabf2e", "val": "82975dfe320d", "test": "2b5aa12bcc8b"}
EXPECTED_LEN = {"train": 27730, "val": 2242, "test": 1856}

def sha256_12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

# 5) Chargement des CSV + controle integrite
final = {}
print("\n=== Chargement des donnees ===")
for n in ["train", "val", "test"]:
    p = CSV_DIR / f"{n}.csv"
    sha = sha256_12(p)
    ok_sha = (sha == EXPECTED_SHA[n])
    final[n] = pd.read_csv(p, encoding="utf-8")
    ok_len = (len(final[n]) == EXPECTED_LEN[n])
    flag = "OK" if (ok_sha and ok_len) else "FAIL"
    print(f"  {n:5s} | {len(final[n]):6d} lignes (attendu {EXPECTED_LEN[n]}) "
          f"| sha256(12) = {sha} [{flag}]")
    assert ok_sha, f"Empreinte {n} incorrecte : attendu {EXPECTED_SHA[n]}, obtenu {sha}"
    assert ok_len,  f"Nb lignes {n} incorrect : attendu {EXPECTED_LEN[n]}, obtenu {len(final[n])}"
    assert list(final[n].columns) == [FON_COL, FR_COL], f"Colonnes inattendues: {list(final[n].columns)}"
    assert not final[n].isna().any().any(), f"Cellules vides dans {n}"
print("  -> Donnees intactes et conformes a la base locale")

# 6) Tokenizer francais (SentencePiece, fige par le groupe)
sp = spm.SentencePieceProcessor(model_file=SPM_PATH)
BOS, EOS = sp.bos_id(), sp.eos_id()
V = sp.get_piece_size()
print(f"\n=== Tokenizer francais (SentencePiece) ===")
print(f"  vocab = {V}  |  pad={sp.pad_id()}  unk={sp.unk_id()}  bos={BOS}  eos={EOS}")
assert V == 4000,         f"Vocab tokenizer inattendu : {V} (attendu 4000)"
assert sp.pad_id() == 0,  "pad_id doit etre 0"
assert sp.unk_id() == 1,  "unk_id doit etre 1"
assert sp.bos_id() == 2,  "bos_id doit etre 2"
assert sp.eos_id() == 3,  "eos_id doit etre 3"

# 7) char2id : reconstruit depuis le train (CHAR_JSON absent)
TOKEN_RE = re.compile(r'[^\s.,;:!?()"«»]+|[.,;:!?()"«»]')
def split_words(s):
    return TOKEN_RE.findall(s)

train_words = [w for s in final["train"][FON_COL] for w in split_words(s)]
char_counts = Counter(c for w in train_words for c in w)
char2id = {c: i + 2 for i, c in enumerate(sorted(char_counts))}
id2char = {i: c for c, i in char2id.items()}
n_chars = len(char2id) + 2  # + PAD et UNK
print(f"\n=== Vocabulaire de caracteres fongbe ===")
print(f"  {len(char2id)} caracteres  (reconstruit depuis train, deterministe)")
print(f"  n_chars (avec PAD, UNK) = {n_chars}")
assert len(char2id) == 107, f"Vocab caracteres inattendu : {len(char2id)} (attendu 107)"

# Sauvegarde pour reproducibilite (dans /kaggle/working)
out_dir = Path("/kaggle/working/artifacts"); out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "char2id.json", "w", encoding="utf-8") as f:
    json.dump(char2id, f, ensure_ascii=False)
print(f"  sauvegarde : {out_dir / 'char2id.json'}")

def word_to_chars(word, max_len=MAX_WORD_LEN):
    ids = [char2id.get(c, UNK) for c in word[:max_len]]
    return ids + [PAD] * (max_len - len(ids))

# 8) Apercu
print("\n=== Apercu (1ere phrase de chaque split) ===")
for n in ["train", "val", "test"]:
    row = final[n].iloc[0]
    print(f"  [{n}] fon : {row[FON_COL][:70]}")
    print(f"        fr  : {row[FR_COL][:70]}")

w = split_words(final["train"][FON_COL][0])[0]
print(f"\n  mot test : {w!r}  ->  ids {word_to_chars(w)[:len(w)]} (puis PAD)")

print(f"\nCellule 1 OK.  device={device}  |  V={V}  |  n_chars={n_chars}  |  pret pour la cellule 2 (datasets/loaders).")

device : cuda (Tesla T4, 15.6 GB)

=== Chargement des donnees ===
  train |  27730 lignes (attendu 27730) | sha256(12) = dac241eabf2e [OK]
  val   |   2242 lignes (attendu 2242) | sha256(12) = 82975dfe320d [OK]
  test  |   1856 lignes (attendu 1856) | sha256(12) = 2b5aa12bcc8b [OK]
  -> Donnees intactes et conformes a la base locale

=== Tokenizer francais (SentencePiece) ===
  vocab = 4000  |  pad=0  unk=1  bos=2  eos=3

=== Vocabulaire de caracteres fongbe ===
  107 caracteres  (reconstruit depuis train, deterministe)
  n_chars (avec PAD, UNK) = 109
  sauvegarde : /kaggle/working/artifacts/char2id.json

=== Apercu (1ere phrase de chaque split) ===
  [train] fon : bo nuɖe tɛ wɛ
        fr  : et qu'est
  [val] fon : ali vɔvɔ ɖokpo lɛ vɔ yi trele.
        fr  : il y a aussitôt une autre voie en terre rouge.
  [test] fon : fitifitì - fitiì
        fr  : embrouillé

  mot test : 'bo'  ->  ids [33, 46] (puis PAD)

Cellule 1 OK.  device=cuda  |  V=4000  |  n_chars=109  |  pret pour la cellu

In [8]:
# ============================================================
# Cellule 2 — Datasets PyTorch + BucketSampler + loaders (P3)
# ============================================================

# Reproductibilité
SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

class PairDataset(Dataset):
    """Pre-tokenise : chaque item = (chars[T,20], tgt_ids[U], ref_str)."""
    def __init__(self, df):
        self.items, self.cut_src, self.cut_tgt = [], 0, 0
        for fo, fr in zip(df[FON_COL], df[FR_COL]):
            words = split_words(fo)
            ids = sp.encode(fr)
            self.cut_src += len(words) > MAX_SRC_W
            self.cut_tgt += len(ids) > MAX_TGT_T - 2
            words, ids = words[:MAX_SRC_W], ids[:MAX_TGT_T - 2]
            if not words:
                continue
            chars = [word_to_chars(w, MAX_WORD_LEN) for w in words]
            self.items.append((chars, [BOS] + ids + [EOS], fr))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    chars, tgts, refs = zip(*batch)
    B, T, L = len(batch), max(len(c) for c in chars), MAX_WORD_LEN
    src = torch.full((B, T, L), PAD, dtype=torch.long)
    for i, c in enumerate(chars):
        src[i, :len(c)] = torch.tensor(c)
    src_len = torch.tensor([len(c) for c in chars])
    U = max(len(t) for t in tgts)
    tgt = torch.full((B, U), PAD, dtype=torch.long)
    for i, t in enumerate(tgts):
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt, list(refs)


class BucketSampler:
    """Regroupe les phrases de longueurs proches pour limiter le padding."""
    def __init__(self, ds, batch_size, shuffle):
        self.ds, self.bs, self.shuffle = ds, batch_size, shuffle
    def __len__(self): return math.ceil(len(self.ds) / self.bs)
    def __iter__(self):
        idx = list(range(len(self.ds)))
        if self.shuffle: random.shuffle(idx)
        chunk, batches = self.bs * 50, []
        for i in range(0, len(idx), chunk):
            part = sorted(idx[i:i + chunk], key=lambda j: len(self.ds.items[j][0]))
            batches += [part[k:k + self.bs] for k in range(0, len(part), self.bs)]
        if self.shuffle: random.shuffle(batches)
        return iter(batches)


# --- Construction ---
BATCH_SIZE = 64
datasets = {n: PairDataset(final[n]) for n in ["train", "val", "test"]}
loaders = {
    n: DataLoader(datasets[n],
                  batch_sampler=BucketSampler(datasets[n], BATCH_SIZE, n == "train"),
                  collate_fn=collate)
    for n in datasets
}

# --- Rapport ---
print("=== Datasets et loaders ===")
for n, ds in datasets.items():
    print(f"  {n:5s} | {len(ds):6d} phrases | src tronquees : {ds.cut_src:3d} | "
          f"tgt tronquees : {ds.cut_tgt:3d} | batches : {len(loaders[n])}")

# --- Controle d'integrite sur un batch ---
src, src_len, tgt, refs = next(iter(loaders["train"]))
print(f"\nBatch echantillon :")
print(f"  src      : {tuple(src.shape)}    (B, T, L=20)")
print(f"  src_len  : min={int(src_len.min())}, max={int(src_len.max())}")
print(f"  tgt      : {tuple(tgt.shape)}    (B, U)")

mot = "".join(id2char[i] for i in src[0, 0].tolist() if i > 1)
cible = sp.decode([i for i in tgt[0].tolist() if i not in (PAD, BOS, EOS)])
print(f"  1er mot  : {mot!r}")
print(f"  cible    : {cible!r}")
print(f"  reference: {refs[0]!r}")
assert cible == refs[0], "Cible reconstruite != reference (pb tokenizer)"

print(f"\nCellule 2 OK.  pret pour la cellule 3 (modeles CharCNN + Encoder + Decoder).")

=== Datasets et loaders ===
  train |  27730 phrases | src tronquees :   0 | tgt tronquees :  95 | batches : 434
  val   |   2242 phrases | src tronquees :   0 | tgt tronquees :  18 | batches : 36
  test  |   1856 phrases | src tronquees :   0 | tgt tronquees :  20 | batches : 29

Batch echantillon :
  src      : (64, 4, 20)    (B, T, L=20)
  src_len  : min=4, max=4
  tgt      : (64, 15)    (B, U)
  1er mot  : 'xo'
  cible    : 'bénir une personne'
  reference: 'bénir une personne'

Cellule 2 OK.  pret pour la cellule 3 (modeles CharCNN + Encoder + Decoder).


In [9]:
# ============================================================
# Cellule 3 — Composants du modèle P3 (CharCNN, Encoder, Attention, Decoder)
# ============================================================

class Highway(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.h, self.g = nn.Linear(d, d), nn.Linear(d, d)
    def forward(self, x):
        g = torch.sigmoid(self.g(x))
        return g * F.relu(self.h(x)) + (1 - g) * x


class CharCNNEmbedding(nn.Module):
    """(B, T, L) indices de caracteres -> (B, T, out_dim) : un vecteur par mot."""
    def __init__(self, n_chars, char_dim=32, widths=(2, 3, 4), n_filters=64, out_dim=256, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(char_dim, n_filters, w) for w in widths])
        d = n_filters * len(widths)
        self.highway = Highway(d)
        self.proj = nn.Linear(d, out_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, chars):
        B, T, L = chars.shape
        x = self.emb(chars.reshape(B * T, L)).transpose(1, 2)               # (B*T, char_dim, L)
        x = torch.cat([F.relu(c(x)).max(dim=2).values for c in self.convs], dim=1)
        x = self.proj(self.highway(x))
        return self.drop(x.view(B, T, -1))


class Encoder(nn.Module):
    """CharCNN -> BiLSTM/BiGRU : lit la phrase FONGBE."""
    def __init__(self, n_chars, rnn_type, d, dropout):
        super().__init__()
        self.is_lstm = rnn_type == "lstm"
        self.embed = CharCNNEmbedding(n_chars, out_dim=d, dropout=dropout)
        self.rnn = (nn.LSTM if self.is_lstm else nn.GRU)(d, d, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.embed(src)                                              # (B, T, d)
        packed = pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)              # (B, T, 2d)
        h, c = hidden if self.is_lstm else (hidden, hidden)              # (2, B, d)
        return self.drop(out), h, c


class BahdanauAttention(nn.Module):
    """A chaque morceau de francais, decide quels mots fongbe regarder."""
    def __init__(self, enc_dim, dec_dim, att_dim):
        super().__init__()
        self.w_enc = nn.Linear(enc_dim, att_dim, bias=False)
        self.w_dec = nn.Linear(dec_dim, att_dim)
        self.v = nn.Linear(att_dim, 1, bias=False)

    def forward(self, s, enc_out, mask):
        e = self.v(torch.tanh(self.w_enc(enc_out) + self.w_dec(s).unsqueeze(1))).squeeze(-1)
        e = e.masked_fill(~mask, float("-inf"))                          # ignorer le padding
        a = torch.softmax(e, dim=-1)
        return torch.bmm(a.unsqueeze(1), enc_out).squeeze(1), a


class Decoder(nn.Module):
    """Ecrit le FRANCAIS, un morceau a la fois."""
    def __init__(self, vocab, rnn_type, d, dropout):
        super().__init__()
        enc = 2 * d
        self.is_lstm = rnn_type == "lstm"
        self.embed = nn.Embedding(vocab, d, padding_idx=PAD)
        self.att = BahdanauAttention(enc, d, d)
        self.cell = (nn.LSTMCell if self.is_lstm else nn.GRUCell)(d + enc, d)
        self.mix = nn.Linear(d + enc + d, d)
        self.proj = nn.Linear(d, vocab)
        self.drop = nn.Dropout(dropout)

    def step(self, y, state, enc_out, mask):
        h, c = state
        e = self.drop(self.embed(y))
        ctx, a = self.att(h, enc_out, mask)
        x = torch.cat([e, ctx], dim=-1)
        if self.is_lstm:
            h, c = self.cell(x, (h, c))
        else:
            h = self.cell(x, h); c = h
        o = torch.tanh(self.mix(torch.cat([h, ctx, e], dim=-1)))
        return self.proj(self.drop(o)), (h, c), a


# --- Test du CharCNN sur GPU ---
char_cnn = CharCNNEmbedding(n_chars).to(device)
n_char_cnn = sum(p.numel() for p in char_cnn.parameters())
print(f"parametres du CharCNN : {n_char_cnn:,}  (attendu : 145632)")
assert n_char_cnn == 145632, "Parametres du CharCNN different du local"

# Smoke test sur un batch de train
src, src_len, tgt, refs = next(iter(loaders["train"]))
src = src.to(device)
char_cnn.eval()
with torch.no_grad():
    out = char_cnn(src)
    seul = char_cnn(src[0:1, 0:1])                       # le meme mot isole

print(f"\nSmoke test CharCNN sur GPU :")
print(f"  entree : {tuple(src.shape)}  -> sortie : {tuple(out.shape)}   (attendu: (B, T, 256))")
print(f"  le vecteur d'un mot ne depend que de ses caracteres : "
      f"{torch.allclose(out[0, 0], seul[0, 0], atol=1e-5)}")
assert out.shape == (src.size(0), src.size(1), 256), "Shape CharCNN incorrecte"
assert torch.allclose(out[0, 0], seul[0, 0], atol=1e-5), "CharCNN depend du contexte (bug)"

# Test rapide de l'encodeur complet (sans encore le Seq2Seq complet)
print(f"\nSmoke test Encoder BiLSTM sur GPU :")
enc = Encoder(n_chars, rnn_type="lstm", d=256, dropout=0.3).to(device)
enc.eval()
with torch.no_grad():
    enc_out, h, c = enc(src, src_len)
print(f"  entree src {tuple(src.shape)} src_len {tuple(src_len.shape)}")
print(f"  -> enc_out {tuple(enc_out.shape)}  (B, T, 512)")
print(f"  -> h {tuple(h.shape)} c {tuple(c.shape)}        (2, B, 256)")
assert enc_out.shape == (src.size(0), src.size(1), 512), "Shape encoder incorrecte"

print(f"\nCellule 3 OK.  pret pour la cellule 4 (Seq2Seq ameliore + scheduler + test surapprentissage).")

parametres du CharCNN : 145,632  (attendu : 145632)

Smoke test CharCNN sur GPU :
  entree : (64, 2, 20)  -> sortie : (64, 2, 256)   (attendu: (B, T, 256))
  le vecteur d'un mot ne depend que de ses caracteres : True

Smoke test Encoder BiLSTM sur GPU :
  entree src (64, 2, 20) src_len (64,)
  -> enc_out (64, 2, 512)  (B, T, 512)
  -> h (2, 64, 256) c (2, 64, 256)        (2, B, 256)

Cellule 3 OK.  pret pour la cellule 4 (Seq2Seq ameliore + scheduler + test surapprentissage).


In [10]:
# ============================================================
# Cellule 4 — Seq2Seq amélioré (weight tying + bridge_c conditionnel) + scheduler + overfit test
# ============================================================
import time
from sacrebleu.metrics import CHRF

# === Modèle amélioré ===
class Seq2Seq(nn.Module):
    """Version livrable :
    - Weight tying : dec.proj.weight = dec.embed.weight (partage, -1M params)
    - bridge_c créé seulement pour LSTM (GRU n'a pas d'état de cellule)
    """
    def __init__(self, n_chars, tgt_vocab, rnn_type="lstm", d=256, dropout=0.3, tying=True):
        super().__init__()
        self.is_lstm = rnn_type == "lstm"
        self.tying = tying
        self.enc = Encoder(n_chars, rnn_type, d, dropout)
        self.bridge_h = nn.Linear(2 * d, d)
        self.bridge_c = nn.Linear(2 * d, d) if self.is_lstm else None
        self.dec = Decoder(tgt_vocab, rnn_type, d, dropout)
        if tying:
            assert self.dec.proj.weight.shape == self.dec.embed.weight.shape, \
                f"shapes {self.dec.proj.weight.shape} vs {self.dec.embed.weight.shape}"
            self.dec.proj.weight = self.dec.embed.weight   # partage par référence

    def encode(self, src, src_len):
        out, h, c = self.enc(src, src_len)
        h0 = torch.tanh(self.bridge_h(torch.cat([h[0], h[1]], dim=-1)))
        if self.is_lstm:
            c0 = torch.tanh(self.bridge_c(torch.cat([c[0], c[1]], dim=-1)))
        else:
            c0 = h0    # placeholder inutilisé par GRUCell
        lengths = src_len.to(out.device)
        mask = torch.arange(out.size(1), device=out.device)[None, :] < lengths[:, None]
        return out, (h0, c0), mask

    def forward(self, src, src_len, tgt):
        """Teacher forcing : entrée tgt[:, :-1], cible tgt[:, 1:]."""
        out, state, mask = self.encode(src, src_len)
        logits = []
        for t in range(tgt.size(1) - 1):
            lg, state, _ = self.dec.step(tgt[:, t], state, out, mask)
            logits.append(lg)
        return torch.stack(logits, dim=1)


def build_scheduler(opt, warmup=500, peak_lr=1e-3):
    """Warmup linéaire (0->peak_lr sur 'warmup' steps) puis décroissance inverse-sqrt.
    Le lr de base dans opt doit être peak_lr (1e-3) — conforme à la règle groupe."""
    def lr_lambda(step):
        s = max(1, step)
        if s < warmup:
            return s / warmup
        return (warmup / s) ** 0.5
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


# === Helpers décodage (pour overfit + entraînement + éval) ===
def ids_to_text(ids):
    out = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): out.append(i)
    return sp.decode(out)


@torch.no_grad()
def greedy_decode(model, src, src_len, max_len=80):
    model.eval()
    out, state, mask = model.encode(src, src_len)
    y = torch.full((src.size(0),), BOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    res = []
    for _ in range(max_len):
        lg, state, _ = model.dec.step(y, state, out, mask)
        y = lg.argmax(-1); res.append(y); done |= (y == EOS)
        if done.all(): break
    return [ids_to_text(r) for r in torch.stack(res, 1).tolist()]


# === Test 1 : param counts + courbe scheduler ===
V = sp.get_piece_size()
print("=== Comptes de paramètres ===\n")
old_counts = {"lstm": 5_023_104, "gru": 4_497_280}
for rnn_type, old in old_counts.items():
    m_test = Seq2Seq(n_chars, V, rnn_type=rnn_type, d=256, dropout=0.3, tying=True)
    n = sum(p.numel() for p in m_test.parameters())
    tag = "A2 (LSTM)" if rnn_type == "lstm" else "A3 (GRU) "
    print(f"  {tag} avec tying : {n:,}   (avant {old:,},  -{old-n:,},  -{(old-n)/old*100:.1f}%)")
    print(f"    weight tying actif : {m_test.dec.proj.weight is m_test.dec.embed.weight}")
    print(f"    bridge_c présent   : {m_test.bridge_c is not None}   (attendu : {rnn_type == 'lstm'})\n")

print("=== Scheduler (warmup=500, peak_lr=1e-3) ===")
def lr_at(step, warmup=500, peak=1e-3):
    s = max(1, step)
    factor = (s / warmup) if s < warmup else (warmup / s) ** 0.5
    return peak * factor
for s in [1, 100, 500, 1000, 2000, 5000, 11000]:
    bar = "#" * int(lr_at(s) / 1e-3 * 30)
    print(f"  step {s:>5} : lr = {lr_at(s):.4e}  {bar}")


# === Test 2 : surapprentissage sur 200 paires (sanity check clé) ===
print("\n=== Test de surapprentissage (200 paires, dropout=0, lr=1e-3) ===")
CFG_OVERFIT = dict(rnn_type="lstm", d=256, dropout=0.0, tying=True)

# 200 paires fixes (4 lots de 64)
fixed, n = [], 0
for s, l, t, _ in loaders["train"]:
    fixed.append((s.to(device), l, t.to(device)))
    n += s.size(0)
    if n >= 200: break
refs_overfit = [ids_to_text(r) for _, _, t in fixed for r in t.tolist()]
print(f"  {n} paires | exemple de référence : {refs_overfit[0]!r}")

@torch.no_grad()
def evaluate_overfit(model):
    hyps = [h for s, l, _ in fixed for h in greedy_decode(model, s, l)]
    return CHRF().corpus_score(hyps, [refs_overfit]).score, hyps

torch.manual_seed(0)
m = Seq2Seq(n_chars, V, **CFG_OVERFIT).to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)

t0 = time.time()
final_ep = 0
final_loss = float("inf")
final_chrf = 0.0
final_hyps = []
for ep in range(1, 201):
    m.train(); tot, ntok = 0.0, 0
    for s, l, t in fixed:
        logits = m(s, l, t)
        gold = t[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, V), gold.reshape(-1),
                               ignore_index=PAD, label_smoothing=0.1)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        with torch.no_grad():
            tot += F.cross_entropy(logits.reshape(-1, V), gold.reshape(-1),
                                   ignore_index=PAD, reduction="sum").item()
            ntok += (gold != PAD).sum().item()
    avg = tot / ntok
    final_ep, final_loss = ep, avg
    if ep == 1 or ep % 10 == 0:
        score, final_hyps = evaluate_overfit(m)
        final_chrf = score
        print(f"  epoch {ep:3d} | loss {avg:.3f} | chrF {score:5.1f} | {time.time()-t0:.0f}s")
    if avg < 0.15:
        score, final_hyps = evaluate_overfit(m)
        final_chrf = score
        break

print(f"\n  FINAL : epoch {final_ep} | loss {final_loss:.3f} | chrF {final_chrf:.1f}")
for i in (0, 1, 2):
    print(f"  réf : {refs_overfit[i]}")
    print(f"  hyp : {final_hyps[i]}\n")

# Assertions de bon fonctionnement
assert final_loss < 0.5, f"Loss ne baisse pas assez ({final_loss:.3f}) — modèle ne surapprend pas"
assert final_chrf > 80, f"chrF trop bas ({final_chrf:.1f}) — modèle ne généralise pas même sur 200 paires"
print("Cellule 4 OK.  Le modèle sait surapprendre -> prêt pour la cellule 5 (entraînement complet A2 LSTM sur GPU).")

=== Comptes de paramètres ===

  A2 (LSTM) avec tying : 3,999,104   (avant 5,023,104,  -1,024,000,  -20.4%)
    weight tying actif : True
    bridge_c présent   : True   (attendu : True)

  A3 (GRU)  avec tying : 3,341,952   (avant 4,497,280,  -1,155,328,  -25.7%)
    weight tying actif : True
    bridge_c présent   : False   (attendu : False)

=== Scheduler (warmup=500, peak_lr=1e-3) ===
  step     1 : lr = 2.0000e-06  
  step   100 : lr = 2.0000e-04  ######
  step   500 : lr = 1.0000e-03  ##############################
  step  1000 : lr = 7.0711e-04  #####################
  step  2000 : lr = 5.0000e-04  ###############
  step  5000 : lr = 3.1623e-04  #########
  step 11000 : lr = 2.1320e-04  ######

=== Test de surapprentissage (200 paires, dropout=0, lr=1e-3) ===
  256 paires | exemple de référence : "obéir à quelqu'un"
  epoch   1 | loss 14.752 | chrF   4.8 | 1s
  epoch  10 | loss 1.901 | chrF   7.9 | 3s
  epoch  20 | loss 0.481 | chrF  53.7 | 6s
  epoch  30 | loss 0.239 | chrF  89

In [14]:
# ============================================================
# Cellule 5 — Entraînement A2 LSTM (GPU Kaggle) SANS scheduler
# ============================================================
import os, csv, time, random
import torch, torch.nn as nn, torch.nn.functional as F
from sacrebleu.metrics import CHRF

# ---------- Réglages ----------
RUN = "A3_gru_P3"
CFG = dict(rnn_type="gru", d=256, dropout=0.3, tying=True)
CKPT_DIR = f"/kaggle/working/checkpoints/{RUN}"
MAX_EPOCHS, EVAL_EVERY, PATIENCE = 30, 500, 3
LR, CLIP, SMOOTH = 1e-3, 1.0, 0.1
QUICK_N = 1000
TIME_BUDGET_MIN = 90
USE_SCHEDULER = False   # <-- False : lr constant 1e-3 (comme le local)

os.makedirs(CKPT_DIR, exist_ok=True)
V = sp.get_piece_size()

# ---------- Validation : pre-chargée sur GPU + sous-ensemble chrF fixe ----------
val_batches = [(s.to(device), l, t.to(device)) for s, l, t, _ in loaders["val"]]
order = list(range(len(val_batches))); random.Random(0).shuffle(order)
quick, n = [], 0
for i in order:
    quick.append(val_batches[i]); n += val_batches[i][0].size(0)
    if n >= QUICK_N: break
quick_refs = [ids_to_text(r) for _, _, t in quick for r in t.tolist()]

@torch.no_grad()
def evaluate():
    model.eval()
    tot, ntok = 0.0, 0
    for s, l, t in val_batches:
        lg = model(s, l, t); gold = t[:, 1:]
        tot += F.cross_entropy(lg.reshape(-1, V), gold.reshape(-1),
                               ignore_index=PAD, reduction="sum").item()
        ntok += (gold != PAD).sum().item()
    hyps = [h for s, l, _ in quick for h in greedy_decode(model, s, l)]
    return tot / ntok, CHRF().corpus_score(hyps, [quick_refs]).score

# ---------- Modèle, optimiseur, reprise ----------
torch.manual_seed(0)
model = Seq2Seq(n_chars, V, **CFG).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
# Pas de scheduler (USE_SCHEDULER = False)

step, epoch, best_val, best_step, bad = 0, 0, float("inf"), 0, 0
last = f"{CKPT_DIR}/last.pt"
if os.path.exists(last):
    ck = torch.load(last, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    step, epoch, best_val, best_step, bad = ck["step"], ck["epoch"], ck["best_val"], ck["best_step"], ck["bad"]
    print(f"REPRISE a l'etape {step} (epoch {epoch}), meilleure val {best_val:.3f}")

def save_ckpt(path):
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                    step=step, epoch=epoch, best_val=best_val, best_step=best_step, bad=bad,
                    config=dict(CFG, n_chars=n_chars, vocab=V, use_scheduler=USE_SCHEDULER),
                    char2id=char2id), path)

log_path = f"{CKPT_DIR}/log.csv"
if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["step", "epoch", "train_loss", "val_loss", "val_chrf", "lr", "session_min"])

n_par = sum(p.numel() for p in model.parameters())
print(f"\n{RUN} | parametres : {n_par:,} | {len(loaders['train'])} lots/epoch "
      f"| eval toutes les {EVAL_EVERY} etapes | lr constant {LR} | budget={TIME_BUDGET_MIN} min\n")

# ---------- Boucle d'entraînement ----------
t0 = time.time(); stop = False; run_loss, run_n, n_sess = 0.0, 0, 0
while epoch < MAX_EPOCHS and not stop:
    epoch += 1; te = time.time(); model.train()
    for s, l, t, _ in loaders["train"]:
        s = s.to(device, non_blocking=True)
        t = t.to(device, non_blocking=True)
        lg = model(s, l, t); gold = t[:, 1:]
        loss = F.cross_entropy(lg.reshape(-1, V), gold.reshape(-1),
                               ignore_index=PAD, label_smoothing=SMOOTH)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        opt.step()                              # <-- pas de sched.step()
        step += 1; n_sess += 1; run_loss += loss.item(); run_n += 1

        if n_sess == 50:
            est = (time.time() - t0) / 50 * len(loaders["train"]) / 60
            print(f"[vitesse] 50 etapes en {time.time()-t0:.0f}s -> environ {est:.1f} min par epoch\n")

        if step % EVAL_EVERY == 0:
            te_eval = time.time()
            vl, vc = evaluate(); model.train()
            tl = run_loss / run_n; run_loss, run_n = 0.0, 0
            mins = (time.time() - t0) / 60
            lr_now = opt.param_groups[0]["lr"]
            improved = vl < best_val
            if improved: best_val, best_step, bad = vl, step, 0
            else: bad += 1
            if improved: save_ckpt(f"{CKPT_DIR}/best.pt")
            save_ckpt(last)
            with open(log_path, "a", newline="") as f:
                csv.writer(f).writerow([step, epoch, round(tl, 4), round(vl, 4),
                                        round(vc, 2), f"{lr_now:.2e}", round(mins, 1)])
            tag = "* meilleur" if improved else f"(sans gain {bad}/{PATIENCE})"
            print(f"etape {step:5d} | epoch {epoch:2d} | train {tl:.3f} | val {vl:.3f} | "
                  f"chrF {vc:4.1f} | lr {lr_now:.2e} | {mins:5.1f} min (eval {time.time()-te_eval:.0f}s) {tag}")
            if bad >= PATIENCE:
                print("-> early stopping"); stop = True; break
            if mins >= TIME_BUDGET_MIN:
                print("-> budget de temps atteint"); stop = True; break
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(f"--- epoch {epoch} : {(time.time()-te)/60:.1f} min")

print(f"\nTERMINE | {RUN} | parametres {n_par:,} | meilleur modele : etape {best_step}, "
      f"val loss {best_val:.3f} | {(time.time()-t0)/60:.0f} min cette session")
print(f"Checkpoints : {CKPT_DIR}/best.pt  +  {CKPT_DIR}/last.pt  +  {CKPT_DIR}/log.csv")


A3_gru_P3 | parametres : 3,341,952 | 434 lots/epoch | eval toutes les 500 etapes | lr constant 0.001 | budget=90 min

[vitesse] 50 etapes en 3s -> environ 0.5 min par epoch

--- epoch 1 : 0.5 min
etape   500 | epoch  2 | train 7.816 | val 6.059 | chrF  9.0 | lr 1.00e-03 |   0.5 min (eval 2s) * meilleur
--- epoch 2 : 0.5 min
etape  1000 | epoch  3 | train 6.072 | val 5.571 | chrF 12.9 | lr 1.00e-03 |   1.1 min (eval 2s) * meilleur
--- epoch 3 : 0.5 min
etape  1500 | epoch  4 | train 5.683 | val 5.278 | chrF 11.4 | lr 1.00e-03 |   1.7 min (eval 2s) * meilleur
--- epoch 4 : 0.5 min
etape  2000 | epoch  5 | train 5.372 | val 5.116 | chrF 12.9 | lr 1.00e-03 |   2.2 min (eval 2s) * meilleur
--- epoch 5 : 0.5 min
etape  2500 | epoch  6 | train 5.173 | val 4.946 | chrF 14.1 | lr 1.00e-03 |   2.8 min (eval 2s) * meilleur
--- epoch 6 : 0.5 min
etape  3000 | epoch  7 | train 4.967 | val 4.832 | chrF 13.0 | lr 1.00e-03 |   3.3 min (eval 2s) * meilleur
--- epoch 7 : 0.5 min
--- epoch 8 : 0.4 min
e

In [15]:
from IPython.display import FileLink

# Télécharger le meilleur modèle
display(FileLink("checkpoints/A3_gru_P3/best.pt"))

# Ou télécharger les logs
display(FileLink("checkpoints/A3_gru_P3/log.csv"))

/kaggle/working/checkpoints/A3_gru_P3/best.pt

/kaggle/working/checkpoints/A3_gru_P3/log.csv